In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from collections import Counter
from torchvision.io import read_image
from collections import defaultdict
import matplotlib.pyplot as plt
import torch.optim as optim
from PIL import Image
import os
import shutil
import time
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, deprocess_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [ ]:
def count_classes(dataset):
    counts= Counter(label for _, label in dataset.samples)
    return counts

In [ ]:
SELECTED_CLASSES= [
    "apple",
    "banana",
    "bell pepper",
    "carrot",
    "grapes",
    "kiwi",
    "lemon",
    "orange",
    "pear",
    "tomato"
]

In [ ]:
import os
!pip install --upgrade kaggle
os.environ["KAGGLE_API_TOKEN"] = "KGAT_2f209c1dc3bf95492f11537f2a7a0fdb"

!mkdir -p ~/.kaggle
!echo $KAGGLE_API_TOKEN > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!kaggle datasets list

In [ ]:
!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition

In [ ]:
!unzip fruit-and-vegetable-image-recognition.zip
!ls

In [ ]:
source_dir = "/content"
target_dir = "/content/Fruit_Dataset_10"

splits = ["train", "validation", "test"]
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

for split in splits:
    split_source = os.path.join(source_dir, split)
    split_target = os.path.join(target_dir, split)

    os.makedirs(split_target, exist_ok=True)

    for class_name in SELECTED_CLASSES:
        class_source = os.path.join(split_source, class_name)
        class_target = os.path.join(split_target, class_name)

        shutil.copytree(class_source, class_target)

In [ ]:
transform= transforms.Compose([
    transforms.Resize((128,128)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor()
])

In [ ]:
train_dataset = ImageFolder("/content/Fruit_Dataset_10/train", transform=transform)
val_dataset = ImageFolder("/content/Fruit_Dataset_10/validation", transform=transform)
test_dataset = ImageFolder("/content/Fruit_Dataset_10/test", transform=transform)

train_counts = list(count_classes(train_dataset).values())
val_counts = list(count_classes(val_dataset).values())
test_counts = list(count_classes(test_dataset).values())

In [ ]:
data= {
    "class": SELECTED_CLASSES,
    "train_counts": train_counts,
    "val_counts": val_counts,
    "test_counts": test_counts
}

df= pd.DataFrame(data)
df

In [ ]:
x= np.arange(len(SELECTED_CLASSES))
w= 0.25
plt.figure(figsize=(12,5))
plt.bar(x-w, train_counts, w, label="Train")
plt.bar(x, val_counts, w, label="Validation")
plt.bar(x + w, test_counts, w, label="Test")
plt.xticks(x, SELECTED_CLASSES, rotation=45, ha="right")
plt.ylabel("Sample count")
plt.title("Class distribution across splits")
plt.legend(); plt.tight_layout(); plt.show()

**The class distribution was analyzed using the number of samples in each class for the training, validation, and test splits. The training set contains between 68 and 100 samples per class, resulting in an imbalance ratio of 1.47 between the largest and smallest classes. Therefore, the dataset can be considered relatively balanced. if it was imbalanced the accuracy for different class could be different because the training size for each one would be different.**

In [ ]:
class_rgb = defaultdict(list)
class_rgb = defaultdict(list)

for index in range(len(train_dataset)):
    image, label= train_dataset[index]
    rgb_mean= image.mean(dim=(1,2))
    rgb_mean= rgb_mean*255
    class_name= train_dataset.classes[label]
    class_rgb[class_name].append(rgb_mean)

In [ ]:
rgb_means = {}
for cls, values in class_rgb.items():
    rgb_means[cls] = torch.stack(values).mean(dim=0)

print(rgb_means["apple"])

In [ ]:
classes= list(rgb_means.keys())

rgb_tensor = torch.stack(
    [rgb_means[c] for c in classes]
)

In [ ]:
x= np.arange(len(classes))
width = 0.25
plt.figure(figsize=(12,5))

plt.bar(
    x-width,
    rgb_tensor[:,0].numpy(),
    width,
    label="R"
)

plt.bar(
    x,
    rgb_tensor[:,1].numpy(),
    width,
    label="G"
)


plt.bar(
    x + width,
    rgb_tensor[:,2].numpy(),
    width,
    label="B"
)

plt.ylabel("Mean RGB intensity")
plt.title("Mean RGB Value per Class")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.scatter(
    rgb_tensor[:,0],
    rgb_tensor[:,1]
)
for i, cls in enumerate(classes):
    plt.text(rgb_tensor[i,0], rgb_tensor[i,1], cls)
plt.xlabel("Red")
plt.ylabel("Green")
plt.title("R vs G")


plt.subplot(1,3,2)
plt.scatter(
    rgb_tensor[:,0],
    rgb_tensor[:,2]
)
for i, cls in enumerate(classes):
    plt.text(rgb_tensor[i,0], rgb_tensor[i,2], cls)
plt.xlabel("Red")
plt.ylabel("Blue")
plt.title("R vs B")


# G vs B
plt.subplot(1,3,3)

plt.scatter(
    rgb_tensor[:,1],
    rgb_tensor[:,2]
)

for i, cls in enumerate(classes):
    plt.text(rgb_tensor[i,1], rgb_tensor[i,2], cls)
plt.xlabel("Green")
plt.ylabel("Blue")
plt.title("G vs B")


plt.tight_layout()
plt.show()

In [ ]:
distance_matrix= torch.cdist(rgb_tensor, rgb_tensor)

plt.figure(figsize=(8,7))
plt.imshow(distance_matrix.numpy())
plt.colorbar()
plt.title("RGB Euclidean Distance Matrix")

plt.show()

most similar fruits are: <br>
1- kiwi and pear <br>
2- banana and lemon <br>
3- banana and pear <br>

because they have the least distace to each other

In [ ]:
all_images= []
for image, _ in train_dataset:
    all_images.append(image)
images= torch.stack(all_images)
mean= images.mean(dim=(0,2,3))
std= images.std(dim=(0,2,3))

In [ ]:
print(mean)
print(std)

**The mean and standard deviation of each RGB channel were calculated exclusively from the training set. These statistics were then used to apply Z-score normalization to the training, validation, and test sets. Computing normalization parameters from the validation or test sets would introduce data leakage because information about unseen data distribution would influence the preprocessing pipeline. This would result in an unrealistic evaluation because the model benefits from information that would not be available during real-world inference.**

In [ ]:
train_transform= transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean,
        std
    )
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [ ]:
train_dataset = ImageFolder("/content/Fruit_Dataset_10/train", transform=train_transform)
val_dataset= ImageFolder("/content/Fruit_Dataset_10/validation", transform=val_transform)
test_dataset= ImageFolder("/content/Fruit_Dataset_10/test", transform=val_transform)

In [ ]:
visual_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    )
])

In [ ]:
samples = train_dataset.samples[:5]
plt.figure(figsize=(10,8))

for i,(path,label) in enumerate(samples):
    image = Image.open(path).convert("RGB")
    augmented = visual_transform(image)
    plt.subplot(5,2,2*i+1)
    plt.imshow(image)
    plt.title("Original: "+train_dataset.classes[label])
    plt.axis("off")

    plt.subplot(5,2,2*i+2)
    plt.imshow(augmented)
    plt.title("Augmented")
    plt.axis("off")
plt.tight_layout()
plt.show()

**Questions** <br>



1- <br>
𝑂𝑢𝑡𝑝𝑢𝑡 𝑆𝑖𝑧𝑒 = [(𝐼𝑛𝑝𝑢𝑡 +2𝑃 − 𝐾)/𝑆] + 1 <br>
1st: [(128+2(2)-5)/1]+1 = 128 <br>
2st: [(128+2(0)-2)/2]+1 = 64 <br>
3st: [(64+2(1)-3)/1]+1 = 64 <br>

2-<br>
Parameters = (Kh * Kw * Cin * Cout) + Cout <br>
1st = 5 * 5 * 3 * 32) + 32 = 2,432 <br>
2st = 0  (pooling has not trainable parameter, it does fixed mathematical operations)<br>
3st = 3 * 3 * 32 * 64) + 64 = 18,496 <br>

3-<br>
𝑅𝐹𝑙 = 𝑅𝐹𝑙−1+(𝐾𝑙 −1)×Π𝑆𝑖 <br>
RF0=1 <br>
1st = 1 + (5-1) * 1 = 5 <br>
2st = 5 + (2-1) * 1 = 6 <br>
3st = 6 + (3-1) * 2 = 10 <br>

4-<br>
If Si changes from 1 to 2, we must recompute the cumulative strides and RF values. RF3 would be 15 instead of 10. generaly it helps converging for %50 because it's like it zoom out. the trade of is between generalization and details. when we increase the stride it will shring the grid size which help us to see more data at each layer but we wanted to detect some small details then bigger stride can cause missing those details.  

In [ ]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=2, stride=2, padding=padding),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size= 2, stride= 2)
        )

    def forward(self, x):
        return self.block(x)

In [ ]:
class ModularCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            CNNBlock(in_channels=3, out_channels=32, padding=2),
            CNNBlock(in_channels=32, out_channels=64, padding=2),
            CNNBlock(in_channels=64, out_channels=128, padding=2),
            CNNBlock(in_channels=128, out_channels=256, padding=2)
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
BATCH_SIZE= 32
train_loader = DataLoader(dataset= train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset= val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(dataset= test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ModularCNN(num_classes=10).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

In [ ]:
max_epochs = 30
early_stop_patience = 5
best_val_loss = float('inf')
patience_counter = 0

history= {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "epoch_times": []}

for epoch in range(max_epochs):
    start= time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total

    model.eval()
    running_val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    epoch_val_loss = running_val_loss / val_total
    epoch_val_acc = val_correct / val_total
    end= time.time()
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_acc"].append(epoch_val_acc)
    history["epoch_times"].append(end-start)

    print(f"Epoch {epoch+1:02d}: Train Loss={epoch_train_loss:.4f} | Val Loss={epoch_val_loss:.4f} | Val Acc={epoch_val_acc:.4f} | epoch time={(end-start):.4f}")

    scheduler.step(epoch_val_loss)
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= early_stop_patience:
            print(f"Early stopping triggered at epoch {epoch+1}!")
            break

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)
early_stop_epoch = len(history["train_loss"])
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(epochs, history["train_loss"],
        label="Train Loss",
        marker="o")

ax.plot(epochs, history["val_loss"],
        label="Validation Loss",
        marker="o")

ax.plot(epochs, history["train_acc"],
        label="Train Accuracy",
        marker="s")

ax.plot(epochs, history["val_acc"],
        label="Validation Accuracy",
        marker="s")

ax.axvline(
    x=early_stop_epoch,
    linestyle="--",
    label=f"Early Stop (Epoch {early_stop_epoch})"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss / Accuracy")
ax.set_title("Training and Validation Metrics")
ax.legend()
ax.grid(True)
plt.show()

avg_epoch_time = np.mean(history["epoch_times"])

print(f"Average training time per epoch: {avg_epoch_time:.2f} seconds")

Looking at the curves, there is no overfitting pattern. overfitting would show training loss falling while validation loss rises. but we can see that validation accuracy consistently exceeds training accuracy throughout training  the training set is fed randomly rotated, flipped, and color-jittered images, making the training task harder. The validation set receives only clean, normalized images, so the model naturally scores higher on it. This gap reflects the difficulty of augmented training, not overfitting. <br>
Based on this trajectory, the validation loss recovered before accumulating 3 consecutive non-improving epochs in every case. The LR scheduler did not activate during this run. the model converged and early-stopped at epoch 14 before the scheduler could trigger.

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)

    return acc, y_true, y_pred

In [ ]:
baseline_acc, y_true_base, y_pred_base = evaluate_model(model, test_loader, device)
print("Baseline Accuracy:", baseline_acc)

In [ ]:
res= classification_report(y_true_base, y_pred_base, target_names=test_dataset.classes)
print(res)

In [ ]:
cm= confusion_matrix(y_true_base, y_pred_base, normalize='true')
disp= ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_dataset.classes)

fig, ax = plt.subplots(figsize=(10,10))
disp.plot(ax=ax)
plt.title("Baseline Confusion Matrix")
plt.show()

the main confusion is on orange-lemon, orange-bellpepper and orange-apple which we didn't predicted.

In [ ]:
gray_transform= transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

gray_test_dataset = ImageFolder("/content/Fruit_Dataset_10/test", transform= gray_transform)
gray_loader= DataLoader(gray_test_dataset, batch_size= BATCH_SIZE, shuffle=False)

In [ ]:
img, _ = gray_test_dataset[0]
plt.imshow(img.permute(1,2,0))
plt.show()

In [ ]:
gray_acc, y_true_gray, y_pred_gray = evaluate_model(model, gray_loader, device)
print(gray_acc)

In [ ]:
class SwapRB:
    def __call__(self, img):
        img = np.array(img)
        img = img[:, :, [2,1,0]]
        return Image.fromarray(img)

In [ ]:
swap_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    SwapRB(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [ ]:
swap_dataset = ImageFolder(
    "/content/Fruit_Dataset_10/test",
    transform=swap_transform
)

swap_loader = DataLoader(
    swap_dataset,
    batch_size=32,
    shuffle=False
)

img, _ = swap_dataset[0]
plt.imshow(img.permute(1,2,0))
plt.show()

In [ ]:
swap_acc, y_true_swap, y_pred_swap = evaluate_model(model, swap_loader, device)
print(swap_acc)

In [ ]:
class AddGaussianNoise:
    def __init__(self, sigma=0.1):
        self.sigma = sigma

    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.sigma
        return torch.clamp(tensor + noise, 0, 1)

In [ ]:
noise_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    AddGaussianNoise(0.1),
    transforms.Normalize(mean, std)
])

In [ ]:
noise_dataset = ImageFolder(
    "/content/Fruit_Dataset_10/test",
    transform=noise_transform
)

noise_loader= DataLoader(noise_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
noise_acc, y_true_noise, y_pred_noise = evaluate_model(model, noise_loader, device)
print(noise_acc)

In [ ]:
class_names = test_dataset.classes
def analyze_test(y_true, y_pred, baseline_acc=None):
    cm = confusion_matrix(y_true, y_pred)
    class_acc = cm.diagonal() / cm.sum(axis=1)

    worst_class_idx = np.argmin(class_acc)
    worst_class = class_names[worst_class_idx]

    cm_no_diag = cm.copy()
    np.fill_diagonal(cm_no_diag, 0)

    i, j = np.unravel_index(
        np.argmax(cm_no_diag),
        cm_no_diag.shape
    )

    confused_pair = f"{class_names[i]} → {class_names[j]}"
    overall_acc = np.trace(cm) / np.sum(cm)

    accuracy_drop= (
        baseline_acc - overall_acc
        if baseline_acc is not None
        else 0
    )

    return {
        "Overall Accuracy": overall_acc,
        "Accuracy Drop": accuracy_drop,
        "Biggest Drop (Class)": worst_class,
        "Most Confused Pair": confused_pair
    }

In [ ]:
results = []

results.append({
    "Test": "Baseline",
    **analyze_test(
        y_true_base,
        y_pred_base
    )
})

results.append({
    "Test": "Grayscale",
    **analyze_test(
        y_true_gray,
        y_pred_gray,
        baseline_acc
    )
})

results.append({
    "Test": "Channel Swap",
    **analyze_test(
        y_true_swap,
        y_pred_swap,
        baseline_acc
    )
})

results.append({
    "Test": "Gaussian Noise",
    **analyze_test(
        y_true_noise,
        y_pred_noise,
        baseline_acc
    )
})

In [ ]:
df = pd.DataFrame(results)
df

The baseline model achieved 74.75% accuracy on the original test set. However, when color information was removed using grayscale conversion, the accuracy dropped significantly to 18%. This indicates that the CNN learned strong color-based features rather than relying mainly on shape and geometry.
The channel swapping experiment produced an even larger degradation, reducing accuracy. This confirms that the model is highly sensitive to color distribution. In contrast, Gaussian noise caused only a moderate accuracy decrease. so we understand that the model relies heavily on color cues for classification. a robust machine should rely on shape and not color. because a dark enviorment color won't help much.

In [ ]:
target_layers = [model.feature_extractor[3].block[0]]
cam = GradCAM(model=model, target_layers=target_layers)

In [ ]:
def generate_gradcam(model, image_tensor, target_class):
    input_tensor = image_tensor.unsqueeze(0).to(device)
    targets= [ClassifierOutputTarget(target_class)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    return grayscale_cam[0]

In [ ]:
def denormalize(img):
    img= img.clone()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    img = img.permute(1,2,0).numpy()
    return np.clip(img, 0, 1)

In [ ]:
def show_gradcam(img_tensor, label):
    rgb_img = denormalize(img_tensor)
    cam_map = generate_gradcam(
        model,
        img_tensor,
        label
    )
    visualization = show_cam_on_image(
        rgb_img,
        cam_map,
        use_rgb=True
    )
    plt.figure(figsize=(6,6))
    plt.imshow(visualization)
    plt.title(f"Class: {test_dataset.classes[label]}")
    plt.axis("off")
    plt.show()

In [ ]:
correct_samples = []
wrong_samples = []
correct_classes = set()
wrong_classes = set()
model.eval()

with torch.no_grad():
    for img, label in test_dataset:
        output = model(img.unsqueeze(0).to(device))
        pred = output.argmax(1).item()
        if (pred == label and label not in correct_classes):
            correct_samples.append((img, label, pred))
            correct_classes.add(label)

        if (pred != label and label not in wrong_classes):
            wrong_samples.append((img, label, pred))
            wrong_classes.add(label)

        if (len(correct_samples) >= 2 and len(wrong_samples) >= 2):
            break

In [ ]:
for img,label,pred in correct_samples:
    show_gradcam(img,pred)

In [ ]:
for img,label,pred in wrong_samples:
    show_gradcam(img,pred)

In [ ]:
img,label = test_dataset[0]
color_cam = generate_gradcam(model, img, label)
gray_img,_ = gray_test_dataset[0]

gray_cam = generate_gradcam(model, gray_img, label)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)

plt.imshow(show_cam_on_image(denormalize(img), color_cam, use_rgb=True))
plt.title("Color Scenario")
plt.axis("off")
plt.subplot(1,2,2)

plt.imshow(show_cam_on_image(denormalize(gray_img),gray_cam,use_rgb=True))
plt.title("Grayscale Scenario")
plt.axis("off")

plt.show()

The Grad-CAM heatmaps for the incorrect predictions show that the model primarily focuses on the fruit itself rather than the image background. Therefore, the errors are likely caused by confusion between visually similar classes rather than spurious background correlations. but if it was we could use those augmentation methods like erasing the back ground to improve the model.